# <span style="color:darkblue"> Project 1 - Scraping the Healthy Food Policy Project Database </span>


<font size = "5">

In this notebook we will ...

<font size = "3">

- Open the policy database from Python using Selenium (Lecture 6c)
- Collect the link to each policy's detail page
- Open each detail page and extract the jurisdiction, state, <br>
policy category, and summary text (Lecture 6c and 6d)
- Store the data in a Pandas DataFrame (Lecture 6d)

Website: https://healthyfoodpolicyproject.org/policy-database

# <span style="color:darkblue"> I. Import Packages </span>

<font size = "5">

Import packages for data processing + web scraping

<font size = "3">

- These are the same packages used in the Lecture 6 script "import_packages.py"

In [13]:
# Manage datasets
import pandas as pd

# Work with time data
import time

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Automate navigating within the browser (Selenium)
from selenium import webdriver

# <span style="color:darkblue"> II. Running Chrome from Python </span>

<font size = "5">

Initialize web driver

<font size = "3">

- This will open a Google Chrome window (do not close it)
- You may have to grant permission to Python to access the browser <br>
via a pop-up window

In [14]:
options = webdriver.ChromeOptions()

driver = webdriver.Chrome(options=options)

<font size = "5">

Define URL and open website

<font size = "3">

- For this to work the browser needs to be open

In [15]:
starting_url = 'https://healthyfoodpolicyproject.org/policy-database'

driver.get(starting_url)

# <span style="color:darkblue"> III. Find the Policy Detail Pages </span>

<font size = "5">

Find the "View Details" links by tag + attribute value

<font size = "3">

- In Developer Tools in Google Chrome, each policy card has a <br>
button container: ``` <div class = "policy-card-button-wrap"> ``` <br>
and the link to the policy is the tag ``` a ``` inside it.
- We use the syntax ``` '//tag[@attribute_name = "attribute_value"]' ``` <br>
and browse to the sub-element with a slash ``` / ```
- Use ```.find_elements()``` (with a plural "s") to find multiple elements

In [16]:
list_links = driver.find_elements('xpath','//div[@class = "policy-card-button-wrap"]/a')

# Count how many elements are extracted
num_links = len(list_links)
print(num_links)

605


<font size = "5">

Extract specific attributes of the HTML tag

<font size = "3">

- Extract the web address of each policy using ``` .get_attribute("href") ```
- We store the addresses in a list <b>before</b> opening any detail page. <br>
Once the browser leaves the listing page, the elements we found above <br>
are no longer valid.

In [17]:
list_urls = [link.get_attribute("href") for link in list_links]
if len(list_urls) != 605:
    raise ValueError(f"Expected 605 policy links, found {len(list_urls)}")

# Display the first policy web addresses
list_urls[0:3]

['https://healthyfoodpolicyproject.org/policy/abingdon-va-code-%c2%a7-18-132-current-through-april-21-2016',
 'https://healthyfoodpolicyproject.org/policy/alachua-county-fla-code-of-ordinances-%c2%a7-407-41-current-as-of-may-8-2019',
 'https://healthyfoodpolicyproject.org/policy/alachua-county-fla-third-amendment-to-emergency-order-no-2020-30-%c2%a7%c2%a7-2-3-july-17-2020']

# <span style="color:darkblue"> IV. Test on a Single Policy </span>

<font size = "5">

Make sure the code works on a single element before using a loop

<font size = "3">

- First we open the detail page of the first policy

In [18]:
try:
    time.sleep(10)
    list_links[0].click()
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    summary_element = soup.select_one(".policy-summary p")
    if summary_element is None:
        raise ValueError("The first policy page has no summary paragraph")
    summary_text = summary_element.get_text(" ", strip=True)
finally:
    driver.quit()

summary_text

'This provision exempts people selling fresh farm products (among other exemptions) from a requirement that mobile peddlers, vendors, and canvassers obtain a permit to operate within the town.'

<font size = "5">

Extract the summary text

<font size = "3">

- The summary is in a paragraph ``` p ``` inside the tag <br>

# <span style="color:darkblue"> V. Scrape and Export All Policy Records </span>

The policy database lists 605 policy detail pages. This section collects jurisdiction name and type, state, food system policy category, summary text, and source URL into a Pandas DataFrame and exports `policy_data_605.csv` in the project folder.

The site's `robots.txt` allows policy-page access and specifies a 10-second crawl delay. The scraper waits at least 10 seconds between detail-page requests, retries transient server/network errors with backoff, and checkpoints every 25 records so a rerun can resume. It keeps categories blank when no category is listed rather than inventing a value. The output reports the number of observations and collection date.

In [21]:
import re
import requests
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

def _policy_field(page_soup, label):
    for item in page_soup.select(".policy-overview-grid-item"):
        heading = item.find("strong")
        if heading and heading.get_text(" ", strip=True) == label:
            value_html = "".join(str(node) for node in heading.next_siblings)
            value = BeautifulSoup(value_html, "html.parser").get_text(" ", strip=True)
            value = re.sub(r"\s*([,;])\s*", r"\1 ", value)
            return value.strip(" ,;")
    return ""

def _policy_record(page_soup, url):
    summary_element = page_soup.select_one(".policy-summary p")
    if summary_element is None:
        raise ValueError(f"No summary paragraph found: {url}")
    record = {
        "jurisdiction": _policy_field(page_soup, "Jurisdiction Name"),
        "jurisdiction_type": _policy_field(page_soup, "Jurisdiction Type"),
        "state": _policy_field(page_soup, "State"),
        "policy_category": _policy_field(page_soup, "Food System Category"),
        "summary": summary_element.get_text(" ", strip=True),
        "url": url,
    }
    missing = [key for key in ("jurisdiction", "jurisdiction_type", "state") if not record[key]]
    if missing:
        raise ValueError(f"Missing {', '.join(missing)} at {url}")
    return record

def _get_policy_page(session, url, last_request_at):
    retryable_statuses = {429, 500, 502, 503, 504}
    for attempt in range(3):
        wait_seconds = 10 - (time.monotonic() - last_request_at)
        if wait_seconds > 0:
            time.sleep(wait_seconds)
        try:
            response = session.get(url, timeout=60)
        except (requests.Timeout, requests.ConnectionError) as error:
            last_request_at = time.monotonic()
            if attempt == 2:
                raise
            retry_seconds = 10 * (2 ** attempt)
            print(f"Temporary request failure for {url}: {error}; retrying in {retry_seconds}s")
            time.sleep(retry_seconds)
            continue

        last_request_at = time.monotonic()
        if response.status_code in retryable_statuses:
            if attempt == 2:
                response.raise_for_status()
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                try:
                    retry_seconds = max(10, float(retry_after))
                except ValueError:
                    retry_at = parsedate_to_datetime(retry_after)
                    if retry_at.tzinfo is None:
                        retry_at = retry_at.replace(tzinfo=timezone.utc)
                    retry_seconds = max(10, (retry_at - datetime.now(timezone.utc)).total_seconds())
            else:
                retry_seconds = 10 * (2 ** attempt)
            print(f"HTTP {response.status_code} for {url}; retrying in {retry_seconds:.0f}s")
            time.sleep(retry_seconds)
            continue

        response.raise_for_status()
        return response, last_request_at

    raise RuntimeError(f"Request retries exhausted for {url}")

if len(list_urls) != 605 or len(set(list_urls)) != 605:
    raise ValueError(f"Expected 605 unique policy URLs, found {len(list_urls)}")

columns = ["jurisdiction", "jurisdiction_type", "state", "policy_category", "summary", "url"]
checkpoint_path = Path.cwd() / "policy_data_605.progress.csv"
if checkpoint_path.exists():
    checkpoint_df = pd.read_csv(checkpoint_path, dtype=str, keep_default_na=False)
    if list(checkpoint_df.columns) != columns:
        raise ValueError(f"Unexpected checkpoint columns in {checkpoint_path}")
    records = checkpoint_df.to_dict(orient="records")
    completed_urls = [record["url"] for record in records]
    if completed_urls != list_urls[:len(records)]:
        raise ValueError(f"Checkpoint does not match the current policy list: {checkpoint_path}")
else:
    previous_records = globals().get("records")
    if previous_records:
        completed_urls = [record["url"] for record in previous_records]
        if completed_urls != list_urls[:len(previous_records)]:
            raise ValueError("In-memory scrape progress does not match the current policy list")
        records = previous_records
    else:
        records = [_policy_record(soup, list_urls[0])]
    pd.DataFrame(records, columns=columns).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (DATASCI530 coursework)"})
last_request_at = time.monotonic()
for index in range(len(records), len(list_urls)):
    url = list_urls[index]
    response, last_request_at = _get_policy_page(session, url, last_request_at)
    page_soup = BeautifulSoup(response.text, "html.parser")
    records.append(_policy_record(page_soup, url))
    if len(records) % 25 == 0 or len(records) == len(list_urls):
        pd.DataFrame(records, columns=columns).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
        print(f"Collected {len(records)} / {len(list_urls)} policies")

policy_df = pd.DataFrame(records, columns=columns)
if len(policy_df) != 605 or policy_df["url"].nunique() != 605:
    raise ValueError(f"Expected 605 unique policies, found {len(policy_df)}")

output_path = Path.cwd() / "policy_data_605.csv"
policy_df.to_csv(output_path, index=False, encoding="utf-8-sig")
checkpoint_path.unlink()
print(f"Collected {len(policy_df)} policies on {datetime.now().date()}.")
print(f"Saved CSV: {output_path.resolve()}")
print(f"Policies without a listed category: {policy_df['policy_category'].eq('').sum()}")
print(f"Policies without source summary text: {policy_df['summary'].eq('').sum()}")

ValueError: Missing jurisdiction_type at https://healthyfoodpolicyproject.org/policy/prince-georges-county-maryland-code-of-ordinances-%c2%a7-12-110c-current-through-jan-6-2020

## Project 1 requirements check

The instructor-approved HFPP source substitutes policy records for the marketplace/product example in the general instructions. The workflow follows the same sequence:

- Open the policy listing and identify the 605 detail-page links.
- Use Selenium to click through to the first detail page and test extraction.
- Loop over the remaining policy pages, extracting jurisdiction name/type, state, food system category, and summary.
- Store the records in a Pandas DataFrame and export the CSV with source URLs.
- Document the collection date, observation count, permission, and the site's 10-second `robots.txt` crawl delay.

These policy fields replace product-specific fields in the general example; blank categories or summaries are retained when the source does not provide them.